In [3]:
#!/usr/bin/env python3
"""
Segment Texas shoreline analysis by inlet-bounded shoreline sections.

Uses:
- place_names.csv to identify inlet/pass boundaries
- tx_shoreline_obs.parquet for raw shoreline observations
- tx_shoreline_stats.parquet for linear/trend statistics
- variance_budget_by_transect.csv for variance-budget results

For each segment between adjacent inlets:
- find nearest transect to each inlet location
- exclude the 3 transects nearest each boundary inlet
- summarize rates and variance metrics
- make heat plots
- plot representative transect time series
- compute a segment-average anomaly time series

Notes
-----
- This version keeps human-readable segment names for titles/tables, but uses
  sanitized filenames for output on Windows.
- It reuses your existing saved results rather than recalculating rates or
  variance budgets.
"""

from __future__ import annotations

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# FILES
# ============================================================

PLACE_NAMES = Path(r"F:/crs/proj/2026_shoreline_analysis/place_names.csv")
TS_FILE = Path(r"F:/crs/proj/2026_shoreline_analysis/processed/tx_shoreline_obs.parquet")
STATS_FILE = Path(r"F:/crs/proj/2026_shoreline_analysis/processed/tx_shoreline_stats.parquet")
VAR_STATS_FILE = Path(r"F:/crs/proj/2026_shoreline_analysis/loess_monthly_basis/variance_budget_by_transect.csv")

OUTDIR = Path(r"F:/crs/proj/2026_shoreline_analysis/segment_analysis_by_inlet")

# ============================================================
# USER SETTINGS
# ============================================================

EXCLUDE_N = 3
N_REP_SERIES = 5
HEATMAP_USE_ANOMALY = True   # True: heat plot of anomaly from transect median
PROFILE_USE_ANOMALY = False  # False: representative profiles plotted as raw shoreline position

# Raw shoreline obs columns
ID_COL = "ID"
DATE_COL = "date"
TIME_COL = "time"
SHORE_COL = "distance"

# Transect location columns
LON_COL = "lon_mid"
LAT_COL = "lat_mid"
S_KM_COL = "s_km"

# From stats file
RATE_OLS_COL = "rate_ols"
RATE_TS_COL = "rate_ts"

# From variance table
VAR_TREND_COL = "var_trend"
VAR_SEASONAL_COL = "var_seasonal"
VAR_RESID_COL = "var_resid"
VAR_TOTAL_COL = "var_total_grid"
PCT_TREND_COL = "pct_trend"
PCT_SEASONAL_COL = "pct_seasonal"
PCT_RESID_COL = "pct_resid"
TREND_DIR_COL = "trend_direction"
CLUSTER_COL = "cluster"

# Place names
PLACE_NAME_COL = "Place_name"
PLACE_LAT_COL = "Latitude"
PLACE_LON_COL = "Longitude"
INLET_COL = "Inlet"


# ============================================================
# HELPERS
# ============================================================

def truthy_inlet(x) -> bool:
    if pd.isna(x):
        return False
    return str(x).strip().upper() == "Y"


def make_safe_filename(text: str, maxlen: int = 80) -> str:
    """
    Convert arbitrary text to a Windows-safe filename stem.
    """
    text = str(text).strip()
    text = re.sub(r'[<>:"/\\|?*]', "_", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace(" ", "_")
    text = re.sub(r"_+", "_", text).strip("_")
    text = text.rstrip(" .")[:maxlen].rstrip(" ._")
    return text if text else "segment"


def haversine_m(lon1, lat1, lon2, lat2):
    """
    Great-circle distance in meters.
    Supports numpy arrays.
    """
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return 6371000.0 * c


def combine_datetime(df: pd.DataFrame, date_col: str = DATE_COL, time_col: str = TIME_COL) -> pd.Series:
    """
    Combine separate date and time columns into one datetime series.
    """
    if date_col in df.columns and time_col in df.columns:
        return pd.to_datetime(df[date_col].astype(str) + " " + df[time_col].astype(str), errors="coerce")
    elif date_col in df.columns:
        return pd.to_datetime(df[date_col], errors="coerce")
    else:
        raise ValueError(f"Need {date_col} and/or {time_col} columns.")


def summarize_series(x: pd.Series, prefix: str) -> dict:
    vals = pd.to_numeric(x, errors="coerce").to_numpy()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
        }
    return {
        f"{prefix}_mean": float(np.nanmean(vals)),
        f"{prefix}_median": float(np.nanmedian(vals)),
        f"{prefix}_std": float(np.nanstd(vals)),
        f"{prefix}_min": float(np.nanmin(vals)),
        f"{prefix}_max": float(np.nanmax(vals)),
    }


def pick_evenly_spaced(values: np.ndarray, n: int) -> np.ndarray:
    values = np.asarray(values)
    if len(values) <= n:
        return values
    ii = np.linspace(0, len(values) - 1, n).round().astype(int)
    return values[ii]


# ============================================================
# LOAD + PREP
# ============================================================

def load_inputs():
    place = pd.read_csv(PLACE_NAMES)
    obs = pd.read_parquet(TS_FILE)
    stats = pd.read_parquet(STATS_FILE)
    var_stats = pd.read_csv(VAR_STATS_FILE)

    obs["datetime"] = combine_datetime(obs, DATE_COL, TIME_COL)

    # unique transect geometry/order from raw obs
    transects_obs = (
        obs[[ID_COL, LON_COL, LAT_COL, S_KM_COL, "coast_order"]]
        .drop_duplicates(subset=[ID_COL])
        .copy()
    )

    # use coast_order if present and valid; otherwise derive from s_km
    if "coast_order" in transects_obs.columns and transects_obs["coast_order"].notna().all():
        transects_obs = transects_obs.sort_values("coast_order").reset_index(drop=True)
    else:
        transects_obs = transects_obs.sort_values(S_KM_COL).reset_index(drop=True)
        transects_obs["coast_order"] = np.arange(len(transects_obs))

    # merge stats + variance results onto unique transects
    stats_keep = [c for c in [ID_COL, RATE_OLS_COL, RATE_TS_COL, LON_COL, LAT_COL, S_KM_COL] if c in stats.columns]
    transects = transects_obs.merge(
        stats[stats_keep],
        on=ID_COL,
        how="left",
        suffixes=("", "_stats"),
    )

    keep_var_cols = [
        ID_COL,
        "s_km_rev",
        "rate_ts",
        TREND_DIR_COL,
        VAR_TREND_COL,
        VAR_SEASONAL_COL,
        VAR_RESID_COL,
        VAR_TOTAL_COL,
        "var_total_raw",
        PCT_TREND_COL,
        PCT_SEASONAL_COL,
        PCT_RESID_COL,
        "tern_x",
        "tern_y",
        CLUSTER_COL,
    ]
    keep_var_cols = [c for c in keep_var_cols if c in var_stats.columns]

    transects = transects.merge(
        var_stats[keep_var_cols],
        on=ID_COL,
        how="left",
        suffixes=("", "_var"),
    )

    return place, obs, stats, var_stats, transects


# ============================================================
# INLET -> NEAREST TRANSECT
# ============================================================

def map_inlets_to_transects(place: pd.DataFrame, transects: pd.DataFrame) -> pd.DataFrame:
    inlets = place.loc[place[INLET_COL].map(truthy_inlet)].copy()
    if inlets.empty:
        raise ValueError("No rows with Inlet == 'Y' found in place_names.csv")

    rows = []
    for _, row in inlets.iterrows():
        d_m = haversine_m(
            row[PLACE_LON_COL],
            row[PLACE_LAT_COL],
            transects[LON_COL].to_numpy(),
            transects[LAT_COL].to_numpy(),
        )
        j = int(np.argmin(d_m))
        tr = transects.iloc[j]

        rows.append({
            "Place_name": row[PLACE_NAME_COL],
            "inlet_lon": row[PLACE_LON_COL],
            "inlet_lat": row[PLACE_LAT_COL],
            "nearest_ID": tr[ID_COL],
            "nearest_lon_mid": tr[LON_COL],
            "nearest_lat_mid": tr[LAT_COL],
            "nearest_s_km": tr[S_KM_COL],
            "nearest_coast_order": tr["coast_order"],
            "distance_to_transect_m": float(d_m[j]),
        })

    inlet_map = pd.DataFrame(rows)

    # If two named inlets map to the same transect, keep the closest one.
    inlet_map = (
        inlet_map.sort_values(["nearest_coast_order", "distance_to_transect_m"])
        .drop_duplicates(subset=["nearest_coast_order"], keep="first")
        .reset_index(drop=True)
    )

    return inlet_map


# ============================================================
# SEGMENTS
# ============================================================

def build_segments(inlet_map: pd.DataFrame, transects: pd.DataFrame, exclude_n: int = EXCLUDE_N) -> pd.DataFrame:
    """
    Build shoreline segments between adjacent inlet boundary transects.
    Excludes the nearest exclude_n transects to each boundary inlet.
    """
    inlet_map = inlet_map.sort_values("nearest_coast_order").reset_index(drop=True)

    rows = []

    for k in range(len(inlet_map) - 1):
        left = inlet_map.iloc[k]
        right = inlet_map.iloc[k + 1]

        left_order = int(left["nearest_coast_order"])
        right_order = int(right["nearest_coast_order"])

        # strict interior between the two inlet-aligned transects
        seg = transects.loc[
            (transects["coast_order"] > left_order) &
            (transects["coast_order"] < right_order)
        ].copy()

        seg = seg.sort_values("coast_order").reset_index(drop=True)

        if len(seg) <= 2 * exclude_n:
            warnings.warn(
                f"Skipping segment {left['Place_name']} -> {right['Place_name']} "
                f"because only {len(seg)} transects lie between the inlets."
            )
            continue

        usable = seg.iloc[exclude_n: len(seg) - exclude_n].copy()
        usable = usable.reset_index(drop=True)

        if usable.empty:
            continue

        segment_name = f"{k+1:02d}_{left['Place_name']}_to_{right['Place_name']}"
        segment_file = make_safe_filename(segment_name, maxlen=80)

        usable["segment_id"] = k + 1
        usable["segment_name"] = segment_name
        usable["segment_file"] = segment_file
        usable["left_inlet"] = left["Place_name"]
        usable["right_inlet"] = right["Place_name"]
        usable["left_boundary_order"] = left_order
        usable["right_boundary_order"] = right_order
        usable["usable_rank"] = np.arange(len(usable))

        rows.append(usable)

    if not rows:
        raise ValueError("No usable shoreline segments were created.")

    return pd.concat(rows, ignore_index=True)


# ============================================================
# TIME-SERIES PRODUCTS
# ============================================================

def build_segment_timeseries(obs: pd.DataFrame, seg_ids: np.ndarray) -> pd.DataFrame:
    seg_obs = obs.loc[obs[ID_COL].isin(seg_ids)].copy()
    seg_obs = seg_obs[[ID_COL, "datetime", SHORE_COL]].copy()
    seg_obs[SHORE_COL] = pd.to_numeric(seg_obs[SHORE_COL], errors="coerce")

    # anomaly relative to each transect median
    med = seg_obs.groupby(ID_COL)[SHORE_COL].median().rename("transect_median")
    seg_obs = seg_obs.merge(med, left_on=ID_COL, right_index=True, how="left")
    seg_obs["shore_anom"] = seg_obs[SHORE_COL] - seg_obs["transect_median"]

    return seg_obs


def build_segment_average(seg_obs: pd.DataFrame) -> pd.DataFrame:
    avg = (
        seg_obs.groupby("datetime")
        .agg(
            seg_mean_anom=("shore_anom", "mean"),
            seg_median_anom=("shore_anom", "median"),
            seg_std_anom=("shore_anom", "std"),
            n_obs=("shore_anom", lambda x: np.sum(np.isfinite(x))),
        )
        .reset_index()
        .sort_values("datetime")
    )
    return avg


def build_heatmap_grid(seg_obs: pd.DataFrame, seg_meta: pd.DataFrame, use_anomaly: bool = True) -> pd.DataFrame:
    tmp = seg_meta[[ID_COL, "usable_rank"]].drop_duplicates()
    dd = seg_obs.merge(tmp, on=ID_COL, how="inner")

    value_col = "shore_anom" if use_anomaly else SHORE_COL
    grid = dd.pivot_table(index="datetime", columns="usable_rank", values=value_col, aggfunc="mean")
    grid = grid.sort_index()

    return grid


# ============================================================
# PLOTTING
# ============================================================

def plot_heatmap(grid: pd.DataFrame, title: str, outfile: Path, label: str):
    fig, ax = plt.subplots(figsize=(11, 6))

    arr = grid.to_numpy()
    im = ax.imshow(
        arr,
        aspect="auto",
        interpolation="none",
        origin="upper",
    )

    ax.set_title(title)
    ax.set_xlabel("Transect order within segment")
    ax.set_ylabel("Time")

    if len(grid.index) > 0:
        yt = np.linspace(0, len(grid.index) - 1, min(8, len(grid.index))).round().astype(int)
        yt = np.unique(yt)
        ax.set_yticks(yt)
        ax.set_yticklabels([pd.to_datetime(grid.index[i]).strftime("%Y-%m-%d") for i in yt])

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(label)

    fig.tight_layout()
    fig.savefig(outfile, dpi=200)
    plt.close(fig)


def plot_profiles(seg_obs: pd.DataFrame, seg_avg: pd.DataFrame, rep_ids: np.ndarray, seg_name: str, outfile: Path):
    fig, ax = plt.subplots(figsize=(12, 5))

    for tid in rep_ids:
        dd = seg_obs.loc[seg_obs[ID_COL] == tid].sort_values("datetime")
        y = dd["shore_anom"] if PROFILE_USE_ANOMALY else dd[SHORE_COL]
        ax.plot(dd["datetime"], y, lw=1.1, alpha=0.75, label=str(tid))

    ax2 = ax.twinx()
    ax2.plot(seg_avg["datetime"], seg_avg["seg_mean_anom"], "k-", lw=2.0, label="segment mean anomaly")
    ax2.plot(seg_avg["datetime"], seg_avg["seg_median_anom"], "k--", lw=1.6, label="segment median anomaly")

    ax.set_title(f"{seg_name}: representative transects and segment-average anomaly")
    ax.set_xlabel("Time")
    ax.set_ylabel("Shoreline anomaly" if PROFILE_USE_ANOMALY else "Shoreline position (distance)")
    ax2.set_ylabel("Segment anomaly (distance)")

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="best", fontsize=8)

    fig.tight_layout()
    fig.savefig(outfile, dpi=200)
    plt.close(fig)


def plot_segment_average(seg_avg: pd.DataFrame, seg_name: str, outfile: Path):
    fig, ax = plt.subplots(figsize=(11, 4))

    ax.plot(seg_avg["datetime"], seg_avg["seg_mean_anom"], lw=2.0, label="mean anomaly")
    ax.plot(seg_avg["datetime"], seg_avg["seg_median_anom"], lw=1.5, label="median anomaly")
    ax.fill_between(
        seg_avg["datetime"],
        seg_avg["seg_mean_anom"] - seg_avg["seg_std_anom"],
        seg_avg["seg_mean_anom"] + seg_avg["seg_std_anom"],
        alpha=0.2,
        label="mean ± std",
    )

    ax.set_title(f"{seg_name}: segment-average shoreline anomaly")
    ax.set_xlabel("Time")
    ax.set_ylabel("Shoreline anomaly (distance)")
    ax.legend()

    fig.tight_layout()
    fig.savefig(outfile, dpi=200)
    plt.close(fig)


# ============================================================
# SEGMENT SUMMARY
# ============================================================

def summarize_segment(seg_meta: pd.DataFrame) -> dict:
    row = {
        "segment_id": int(seg_meta["segment_id"].iloc[0]),
        "segment_name": seg_meta["segment_name"].iloc[0],
        "segment_file": seg_meta["segment_file"].iloc[0],
        "left_inlet": seg_meta["left_inlet"].iloc[0],
        "right_inlet": seg_meta["right_inlet"].iloc[0],
        "n_transects": int(seg_meta[ID_COL].nunique()),
        "s_km_min": float(seg_meta[S_KM_COL].min()),
        "s_km_max": float(seg_meta[S_KM_COL].max()),
    }

    for col in [
        RATE_OLS_COL,
        RATE_TS_COL,
        VAR_TREND_COL,
        VAR_SEASONAL_COL,
        VAR_RESID_COL,
        VAR_TOTAL_COL,
        PCT_TREND_COL,
        PCT_SEASONAL_COL,
        PCT_RESID_COL,
    ]:
        if col in seg_meta.columns:
            row.update(summarize_series(seg_meta[col], col))

    if TREND_DIR_COL in seg_meta.columns:
        row["trend_direction_mean"] = float(pd.to_numeric(seg_meta[TREND_DIR_COL], errors="coerce").mean())

    if CLUSTER_COL in seg_meta.columns:
        mode_vals = pd.to_numeric(seg_meta[CLUSTER_COL], errors="coerce").dropna()
        row["cluster_mode"] = float(mode_vals.mode().iloc[0]) if len(mode_vals) else np.nan

    return row


# ============================================================
# MAIN
# ============================================================

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)

    place, obs, stats, var_stats, transects = load_inputs()

    inlet_map = map_inlets_to_transects(place, transects)
    inlet_map.to_csv(OUTDIR / "inlet_to_nearest_transect.csv", index=False)

    seg_members = build_segments(inlet_map, transects, exclude_n=EXCLUDE_N)
    seg_members.to_csv(OUTDIR / "segment_membership.csv", index=False)

    summary_rows = []

    for seg_name, seg_meta in seg_members.groupby("segment_name"):
        seg_file = seg_meta["segment_file"].iloc[0]
        seg_ids = seg_meta[ID_COL].drop_duplicates().to_numpy()

        seg_obs = build_segment_timeseries(obs, seg_ids)
        seg_avg = build_segment_average(seg_obs)

        rep_ids = pick_evenly_spaced(
            seg_meta.sort_values("usable_rank")[ID_COL].drop_duplicates().to_numpy(),
            N_REP_SERIES,
        )

        # save per-segment member table
        seg_meta.sort_values("usable_rank").to_csv(
            OUTDIR / f"{seg_file}_transects.csv", index=False
        )

        # save per-segment average series
        seg_avg.to_csv(
            OUTDIR / f"{seg_file}_segment_average_timeseries.csv", index=False
        )

        # heat plot
        grid = build_heatmap_grid(seg_obs, seg_meta, use_anomaly=HEATMAP_USE_ANOMALY)
        plot_heatmap(
            grid,
            title=f"{seg_name}: shoreline heat plot",
            outfile=OUTDIR / f"{seg_file}_heatplot.png",
            label="Shoreline anomaly (distance)" if HEATMAP_USE_ANOMALY else "Shoreline position (distance)",
        )

        # representative profiles
        plot_profiles(
            seg_obs,
            seg_avg,
            rep_ids,
            seg_name,
            OUTDIR / f"{seg_file}_profiles.png",
        )

        # segment-average plot
        plot_segment_average(
            seg_avg,
            seg_name,
            OUTDIR / f"{seg_file}_segment_average.png",
        )

        # summary row
        row = summarize_segment(seg_meta)
        row["rep_ids"] = ",".join(map(str, rep_ids))
        summary_rows.append(row)

    segment_summary = pd.DataFrame(summary_rows)
    segment_summary.to_csv(OUTDIR / "segment_summary_report.csv", index=False)

    print(f"Wrote outputs to: {OUTDIR}")
    print("Files include:")
    print("  inlet_to_nearest_transect.csv")
    print("  segment_membership.csv")
    print("  segment_summary_report.csv")
    print("  *_transects.csv")
    print("  *_segment_average_timeseries.csv")
    print("  *_heatplot.png")
    print("  *_profiles.png")
    print("  *_segment_average.png")


if __name__ == "__main__":
    main()

Wrote outputs to: F:\crs\proj\2026_shoreline_analysis\segment_analysis_by_inlet
Files include:
  inlet_to_nearest_transect.csv
  segment_membership.csv
  segment_summary_report.csv
  *_transects.csv
  *_segment_average_timeseries.csv
  *_heatplot.png
  *_profiles.png
  *_segment_average.png
